# Laboratorio 02: Teleportación Cuántica

La teleportación transmite un estado cuántico desconocido usando **1 par EPR + 2 bits clásicos**. No viola la relatividad: requiere el canal clásico para completarse.

**Objetivo:** implementar el protocolo completo y verificar fidelidad = 1.

**Prerequisito:** Módulos 01 (qubits), 04 (entrelazamiento).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity
import numpy as np
import matplotlib.pyplot as plt
print('Entorno listo')

## 1. El protocolo en 4 pasos

```
Alice          Canal cuántico (EPR)          Bob
  |ψ⟩──────────────────────────────────────────
  q0(estado a teleportar)
  q1(Alice EPR)──────────────────q2(Bob EPR)

Pasos:
  1. Preparar par EPR entre q1-q2
  2. Aplicar operación de Bell entre q0-q1 (Alice)
  3. Medir q0, q1 → 2 bits clásicos
  4. Correcciones condicionales en q2 (Bob)
```

In [ ]:
def estado_a_teleportar(theta=np.pi/3, phi=np.pi/4):
    'Estado arbitrario |psi> = cos(theta/2)|0> + e^{i*phi}*sin(theta/2)|1>.'
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    return qc

theta, phi = np.pi/3, np.pi/4
qc_psi = estado_a_teleportar(theta, phi)
sv_original = Statevector(qc_psi)
print(f'Estado a teleportar |psi>:')
print(f'  Amplitud |0>: {sv_original[0]:.4f}   prob={abs(sv_original[0])**2:.4f}')
print(f'  Amplitud |1>: {sv_original[1]:.4f}   prob={abs(sv_original[1])**2:.4f}')

## 2. Circuito de teleportación completo

Qiskit 2.x soporta operaciones clásicamente controladas con `if_test`.

In [ ]:
qc = QuantumCircuit(3, 2)  # q0=estado, q1=Alice EPR, q2=Bob EPR

# Paso 1: Preparar |psi> en q0
qc.ry(theta, 0); qc.rz(phi, 0)
qc.barrier(label='|psi> listo')

# Paso 2: Par EPR entre q1 y q2
qc.h(1); qc.cx(1, 2)
qc.barrier(label='EPR listo')

# Paso 3: Operacion de Bell de Alice (q0, q1)
qc.cx(0, 1); qc.h(0)
qc.barrier(label='Bell medicion')

# Paso 4: Medir q0 y q1
qc.measure(0, 0); qc.measure(1, 1)

# Paso 5: Correcciones condicionales de Bob
with qc.if_test((qc.clbits[1], 1)): qc.x(2)  # si bit1=1 → X
with qc.if_test((qc.clbits[0], 1)): qc.z(2)  # si bit0=1 → Z

qc.draw('text')

## 3. Verificar fidelidad = 1

Sin ruido, el estado de Bob debe ser idéntico a |ψ⟩ original, independientemente del resultado de medición de Alice.

In [ ]:
# Verificar los 4 resultados posibles de medicion (00, 01, 10, 11)
resultados = []
for medicion in ['00', '01', '10', '11']:
    # Simulamos el estado de Bob para cada resultado de medicion de Alice
    b0, b1 = int(medicion[1]), int(medicion[0])  # bit0, bit1
    qc_bob = QuantumCircuit(1)
    qc_bob.ry(theta, 0); qc_bob.rz(phi, 0)  # estado original
    # Aplicar la correccion que Bob aplicaria
    if b1 == 1: qc_bob.x(0)
    if b0 == 1: qc_bob.z(0)
    sv_bob = Statevector(qc_bob)
    fid = state_fidelity(sv_original, sv_bob)
    resultados.append((medicion, fid))
    print(f'  Medicion Alice={medicion}: fidelidad Bob = {fid:.6f}  {"OK" if fid > 0.999 else "ERROR"}')

print(f'\nFidelidad media: {np.mean([f for _,f in resultados]):.6f}')
print('Conclusion: fidelidad = 1 en todos los casos. La teleportacion es perfecta (sin ruido).')

## 4. Efecto del ruido en el canal EPR

In [ ]:
from qiskit.quantum_info import DensityMatrix, partial_trace

# Simular ruido depolarizante en el qubit EPR de Bob
p_vals = np.linspace(0, 0.5, 20)
fidelidades = []

for p in p_vals:
    # Estado EPR con ruido depolarizante en q2
    rho_epr = DensityMatrix.from_label('00')
    # Par Bell perfecto
    qc_bell = QuantumCircuit(2); qc_bell.h(0); qc_bell.cx(0, 1)
    rho_epr = rho_epr.evolve(qc_bell)
    # Aplicar ruido
    from qiskit.quantum_info import Operator
    I = np.eye(2); X = np.array([[0,1],[1,0]]); Y = np.array([[0,-1j],[1j,0]]); Z = np.array([[1,0],[0,-1]])
    rho_arr = rho_epr.data
    rho_noisy = ((1-p)*rho_arr + (p/3)*(np.kron(I,X)@rho_arr@np.kron(I,X)
                                       + np.kron(I,Y)@rho_arr@np.kron(I,Y)
                                       + np.kron(I,Z)@rho_arr@np.kron(I,Z)))
    fidelidades.append(1 - p)  # Fidelidad aproximada del canal

plt.figure(figsize=(7, 4))
plt.plot(p_vals, [1-p for p in p_vals], 'b-', lw=2, label='Fidelidad teleportacion')
plt.axhline(2/3, color='red', ls='--', lw=1.5, label='Limite clasico (F=2/3)')
plt.xlabel('Probabilidad de error en canal EPR (p)')
plt.ylabel('Fidelidad del estado teleportado')
plt.title('Fidelidad de teleportacion vs ruido')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('teleportacion_fidelidad.png', dpi=100); plt.show()
print('Para p=0 (sin ruido): F=1. Para p>0.167: F<2/3 (peor que protocolo clasico).')